<a href="https://colab.research.google.com/github/Jericho-Ram/Leadgen/blob/main/leadgen_pipeline_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lead Generation Pipeline — v1.0

**Owner:** Edo Sanjaya P (Jericho) · **Runtime:** Google Colab (free tier)

A six-stage pipeline that turns a raw company list into a scored, verified,
client-deliverable lead sheet.

| Stage | What it does | Output |
|---|---|---|
| 1. Ingest | Load raw leads from Sheets / CSV / Drive | `df_raw` |
| 2. Clean | Normalise, extract root domain, dedupe | `df_clean` |
| 3. Enrich | Fetch site, detect language + industry signals | `df_enriched` |
| 4. Verify | Email format + MX record check | `df_verified` |
| 5. Score | Transparent 100-point ICP rubric | `df_scored` |
| 6. Export | Multi-tab XLSX + optional Google Sheet | deliverable file |

**Design rules this notebook follows**
- Every knob lives in `CONFIG` (Cell 2). No magic numbers buried in functions.
- Scoring is *reason-before-score*: every point is traceable to a stated reason.
- Nothing is invented. Unknown fields stay `None`, never guessed.
- Network work checkpoints to Drive, so a Colab disconnect costs you minutes, not hours.
- `robots.txt` is checked before any page fetch.

**Run order:** Cell 1 → Cell 2 → ... top to bottom. Each stage is independently
re-runnable once the stage before it has produced its dataframe.


---
## Cell 1 — Install dependencies

Run once per Colab session (Colab wipes the VM when it disconnects).
Takes about 40 seconds.


In [4]:
# --- Cell 1: dependencies -----------------------------------------------
%pip install -q tldextract dnspython openpyxl gspread-dataframe tqdm requests beautifulsoup4 lxml

print("Dependencies installed. Restart NOT required.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 13.4 MB/s eta 0:00:00
Dependencies installed. Restart NOT required.


---
## Cell 2 — CONFIG

**This is the only cell you edit for a new client.** Everything downstream reads
from here.

Three things to set before your first run:
1. `INPUT_MODE` — where your raw leads come from.
2. `COLUMN_MAP` — map your source file's column names to the pipeline's names.
3. `ICP` — who counts as a good lead for this client.


In [5]:
# --- Cell 2: CONFIG ------------------------------------------------------
CONFIG = {

    # ---------- CLIENT / RUN IDENTITY ----------
    "client_name":   "Demo Client",
    "campaign_name":  "apac_saas_q3",
    "run_notes":      "First run. Source list from AroundDeal export.",

    # ---------- INPUT ----------
    # One of: "sample" | "csv_upload" | "drive_csv" | "google_sheet"
    "INPUT_MODE": "sample",

    "drive_csv_path":  "/content/drive/MyDrive/leadgen/raw_leads.csv",
    "google_sheet_id": "",          # the long id in the Sheet URL
    "google_sheet_tab": "Raw",

    # Map YOUR source columns -> pipeline names. Left = pipeline, right = yours.
    # Set a value to None if your source does not have that column.
    "COLUMN_MAP": {
        "company_name": "Company",
        "website":      "Website",
        "contact_name": "Contact Name",
        "job_title":    "Title",
        "email":        "Email",
        "country":      "Country",
        "city":         "City",
        "employee_count": "Employees",
        "industry":     "Industry",
        "linkedin_url": "LinkedIn",
    },

    # ---------- IDEAL CUSTOMER PROFILE ----------
    "ICP": {
        # Full points. Matched on WORD BOUNDARIES, so "ceo" will not fire on
        # "ceonomics" and "intern" will not fire on "internal audit".
        # Include spelled-out variants: exports are wildly inconsistent about
        # whether they say "CEO" or "Chief Executive Officer".
        "decision_maker_titles": [
            "founder", "co-founder", "cofounder",
            "ceo", "cto", "coo", "cmo", "cfo", "cro",
            "chief executive", "chief technology", "chief operating",
            "chief marketing", "chief financial", "chief revenue",
            "owner", "proprietor", "president", "director", "head of",
            "vp", "svp", "evp", "vice president", "partner",
            "managing director", "managing partner", "general manager",
            "principal",
            # Chinese titles — matched as substrings (CJK has no word boundaries).
            # Longer titles must come before shorter ones they contain.
            "首席执行官", "执行长", "董事长", "董事總經理", "总经理", "總經理",
            "创始人", "創辦人", "创办人", "总裁", "總裁", "负责人", "負責人",
            "合伙人", "合夥人", "总监", "總監",
        ],
        # Half points. Real influence, rarely final budget authority.
        "influencer_titles": [
            "manager", "lead", "supervisor", "specialist",
            "senior", "coordinator", "officer", "consultant",
            "经理", "經理", "主管", "主任", "专员", "專員",
        ],
        # Zero points, checked first. Students, job seekers, gatekeepers.
        "excluded_titles": [
            "intern", "student", "trainee", "volunteer", "retired",
            "assistant to", "executive assistant", "seeking", "unemployed",
        ],

        # Employee band that fits the offer.
        "employee_min": 10,
        "employee_max": 500,

        # Industry / positioning keywords looked for on the company website.
        "industry_keywords": [
            "saas", "software", "platform", "b2b", "logistics",
            "manufacturing", "export", "wholesale", "distribution",
        ],

        # Countries that count as in-market. ISO-ish names, lowercase.
        "target_countries": [
            "singapore", "malaysia", "indonesia", "taiwan", "hong kong",
            "china", "thailand", "vietnam", "philippines", "japan",
        ],

        # Your differentiator: markets where Mandarin capability is an edge.
        "mandarin_markets": ["china", "taiwan", "hong kong", "singapore", "macau"],
    },

    # ---------- SCORING WEIGHTS (must total 100) ----------
    "WEIGHTS": {
        "email_verified":     30,   # deliverable email = the thing you sell
        "decision_maker":     20,   # right person
        "company_size_fit":   15,   # right size
        "industry_match":     15,   # right business
        "apac_mandarin_edge": 10,   # your differentiator
        "website_live":       10,   # reachable, real company
    },

    # Tier cutoffs applied to the final 0-100 score.
    "TIERS": {"A": 75, "B": 55, "C": 35},   # below C -> "D"

    # ---------- NETWORK BEHAVIOUR ----------
    "REQUEST_DELAY_SEC":  2.0,   # politeness gap between site fetches
    "REQUEST_TIMEOUT":    10,
    "MAX_RETRIES":        2,
    "RESPECT_ROBOTS":     True,  # leave True for anything client-facing
    "USER_AGENT": "Mozilla/5.0 (compatible; LeadResearchBot/1.0; +research contact)",
    "ENRICH_LIMIT": None,        # e.g. 25 to test on a slice first; None = all

    # ---------- CHECKPOINTING ----------
    "USE_DRIVE":        False,   # True = survive disconnects, needs Drive mount
    "CHECKPOINT_DIR":   "/content/drive/MyDrive/leadgen/checkpoints",
    "LOCAL_CHECKPOINT_DIR": "/content/checkpoints",

    # ---------- OUTPUT ----------
    "OUTPUT_DIR":       "/content/output",
    "WRITE_TO_SHEET":   False,   # True = also push to a Google Sheet
    "OUTPUT_SHEET_ID":  "",
}

# --- integrity checks: fail loudly now rather than silently later ----------
_w = sum(CONFIG["WEIGHTS"].values())
assert _w == 100, f"WEIGHTS must total 100, currently {_w}"
assert CONFIG["TIERS"]["A"] > CONFIG["TIERS"]["B"] > CONFIG["TIERS"]["C"], \
    "Tier cutoffs must descend"
assert CONFIG["INPUT_MODE"] in {"sample", "csv_upload", "drive_csv", "google_sheet"}, \
    "Unknown INPUT_MODE"

print(f"Config OK — client: {CONFIG['client_name']} | campaign: {CONFIG['campaign_name']}")
print(f"Weights total: {_w} | Input mode: {CONFIG['INPUT_MODE']}")


Config OK — client: Demo Client | campaign: apac_saas_q3
Weights total: 100 | Input mode: sample


---
## Cell 3 — Imports and helpers

Sets up logging, the checkpoint system, and the shared HTTP session. Nothing
client-specific here — you should never need to edit this cell.


In [6]:
# --- Cell 3: imports + shared helpers ------------------------------------
import os, re, json, time, socket, logging, warnings
from datetime import datetime, timezone
from urllib.parse import urlparse, urlunparse
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup
import tldextract
import dns.resolver
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("leadgen")

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
log.info(f"Run id: {RUN_ID}")


# ---------- checkpointing ----------
def _ckpt_dir():
    d = CONFIG["CHECKPOINT_DIR"] if CONFIG["USE_DRIVE"] else CONFIG["LOCAL_CHECKPOINT_DIR"]
    os.makedirs(d, exist_ok=True)
    return d

def save_checkpoint(df, name):
    # Persist a stage output so a Colab disconnect does not cost you the work.
    path = os.path.join(_ckpt_dir(), f"{CONFIG['campaign_name']}__{name}.parquet")
    df.to_parquet(path, index=False)
    log.info(f"Checkpoint saved: {path} ({len(df)} rows)")
    return path

def load_checkpoint(name):
    path = os.path.join(_ckpt_dir(), f"{CONFIG['campaign_name']}__{name}.parquet")
    if os.path.exists(path):
        df = pd.read_parquet(path)
        log.info(f"Checkpoint loaded: {name} ({len(df)} rows)")
        return df
    log.info(f"No checkpoint found for '{name}'")
    return None


# ---------- HTTP ----------
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": CONFIG["USER_AGENT"],
    "Accept-Language": "en,zh;q=0.8,id;q=0.7",
})

_ROBOTS_CACHE = {}

def robots_allows(url):
    # Check robots.txt once per domain, then cache the verdict.
    if not CONFIG["RESPECT_ROBOTS"]:
        return True
    try:
        p = urlparse(url)
        base = f"{p.scheme}://{p.netloc}"
        if base not in _ROBOTS_CACHE:
            rp = RobotFileParser()
            rp.set_url(base + "/robots.txt")
            try:
                rp.read()
                _ROBOTS_CACHE[base] = rp
            except Exception:
                _ROBOTS_CACHE[base] = None      # unreachable robots -> allow
        rp = _ROBOTS_CACHE[base]
        if rp is None:
            return True
        return rp.can_fetch(CONFIG["USER_AGENT"], url)
    except Exception:
        return True

def fetch(url):
    # Polite GET with retries. Returns (html, status_note).
    if not robots_allows(url):
        return None, "blocked_by_robots"
    last = "unknown_error"
    for attempt in range(CONFIG["MAX_RETRIES"] + 1):
        try:
            r = SESSION.get(url, timeout=CONFIG["REQUEST_TIMEOUT"], allow_redirects=True)
            if r.status_code == 200:
                return r.text, "ok"
            last = f"http_{r.status_code}"
            if 400 <= r.status_code < 500:
                break                     # client errors will not fix themselves
        except requests.exceptions.SSLError:
            last = "ssl_error"; break
        except requests.exceptions.ConnectTimeout:
            last = "timeout"
        except requests.exceptions.ConnectionError:
            last = "connection_error"
        except Exception as e:
            last = f"error_{type(e).__name__}"
        time.sleep(1.5 * (attempt + 1))
    return None, last

print("Helpers ready.")


Helpers ready.


---
## Cell 4 — Stage 1: Ingest

Loads raw leads and applies `COLUMN_MAP`. If `INPUT_MODE` is `"sample"` this
generates a small realistic dataset so you can run the whole notebook end to end
before touching client data — including deliberately broken rows, so you can see
how the pipeline handles them.


In [7]:
# --- Cell 4: Stage 1 — Ingest --------------------------------------------
PIPELINE_COLS = list(CONFIG["COLUMN_MAP"].keys())

def _sample_data():
    # Deliberately messy: duplicates, bad emails, dead domains, missing fields.
    rows = [
        ("Rui Feng Logistics", "https://www.ruifeng-logistics.com", "Chen Wei", "Chief Executive Officer", "chen.wei@ruifeng-logistics.com", "Taiwan", "Taipei", 120, "Logistics", ""),
        ("Rui Feng Logistics Co Ltd", "http://ruifeng-logistics.com/", "Chen Wei", "CEO", "chen.wei@ruifeng-logistics.com", "Taiwan", "Taipei", 120, "Logistics", ""),
        ("Meridian SaaS Pte Ltd", "https://meridian-saas.example.sg", "Aisyah Rahman", "Head of Growth", "aisyah@meridian-saas.example.sg", "Singapore", "Singapore", 45, "Software", ""),
        ("Nusantara Export", "https://nusantaraexport.co.id", "Budi Santoso", "Owner", "budi@nusantaraexport.co.id", "Indonesia", "Surabaya", 30, "Export", ""),
        ("Bright Path Interns", "https://brightpath.example.com", "Sam Lee", "Marketing Intern", "sam@brightpath.example.com", "Malaysia", "Penang", 8, "Agency", ""),
        ("Global Mega Corp", "https://globalmega.example.com", "Jane Doe", "Director of Operations", "jane.doe@globalmega.example.com", "United States", "Austin", 12000, "Manufacturing", ""),
        ("Hong Kong Trade Partners", "https://hktradepartners.example.hk", "Lam Ka Yiu", "Managing Director", "lam@hktradepartners.example.hk", "Hong Kong", "Kowloon", 65, "Distribution", ""),
        ("Broken Email Ltd", "https://broken-email.example.com", "No Name", "COO", "not-an-email", "Vietnam", "Hanoi", 90, "Software", ""),
        ("No Website Sdn Bhd", "", "Tan Mei Ling", "General Manager", "tan@nowebsite.example.my", "Malaysia", "Johor", 55, "Wholesale", ""),
        ("Dead Domain Industries", "https://this-domain-does-not-resolve-xyz123.com", "Ghost User", "CTO", "ghost@this-domain-does-not-resolve-xyz123.com", "Thailand", "Bangkok", 200, "Manufacturing", ""),
        ("上海精密制造有限公司", "https://shanghai-precision.example.cn", "Zhang Wei", "总经理", "zhang.wei@shanghai-precision.example.cn", "China", "Shanghai", 180, "Manufacturing", ""),
        ("Taipei Software Group", "https://taipei-soft.example.tw", "Lin Yi Chen", "C.E.O.", "lin@taipei-soft.example.tw", "Taiwan", "Taipei", 75, "Software", ""),
    ]
    cols = ["company_name","website","contact_name","job_title","email",
            "country","city","employee_count","industry","linkedin_url"]
    return pd.DataFrame(rows, columns=cols)


def ingest():
    mode = CONFIG["INPUT_MODE"]

    if mode == "sample":
        df = _sample_data()
        log.info(f"Sample data loaded: {len(df)} rows (includes intentional bad rows)")
        return df

    if mode == "csv_upload":
        from google.colab import files
        up = files.upload()                       # opens a file picker
        fname = list(up.keys())[0]
        df = pd.read_csv(fname)

    elif mode == "drive_csv":
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        df = pd.read_csv(CONFIG["drive_csv_path"])

    elif mode == "google_sheet":
        from google.colab import auth
        import gspread
        from google.auth import default
        auth.authenticate_user()
        creds, _ = default()
        gc = gspread.authorize(creds)
        ws = gc.open_by_key(CONFIG["google_sheet_id"]).worksheet(CONFIG["google_sheet_tab"])
        df = pd.DataFrame(ws.get_all_records())

    # ---- apply COLUMN_MAP ----
    out = pd.DataFrame()
    missing = []
    for pipeline_col, source_col in CONFIG["COLUMN_MAP"].items():
        if source_col and source_col in df.columns:
            out[pipeline_col] = df[source_col]
        else:
            out[pipeline_col] = None
            if source_col:
                missing.append(f"{pipeline_col} <- '{source_col}'")
    if missing:
        log.warning("Columns not found in source, filled with None: " + ", ".join(missing))
    log.info(f"Ingested {len(out)} rows from {mode}")
    return out


df_raw = ingest()
df_raw["_source_row"] = range(len(df_raw))
save_checkpoint(df_raw, "01_raw")
display(df_raw.head(10))
print(f"\nShape: {df_raw.shape}")


,company_name,website,contact_name,job_title,email,country,city,employee_count,industry,linkedin_url,_source_row
0,Rui Feng Logistics,https://www.ruifeng-logistics.com,Chen Wei,Chief Executive Officer,chen.wei@ruifeng-logistics.com,Taiwan,Taipei,120,Logistics,,0
1,Rui Feng Logistics Co Ltd,http://ruifeng-logistics.com/,Chen Wei,CEO,chen.wei@ruifeng-logistics.com,Taiwan,Taipei,120,Logistics,,1
2,Meridian SaaS Pte Ltd,https://meridian-saas.example.sg,Aisyah Rahman,Head of Growth,aisyah@meridian-saas.example.sg,Singapore,Singapore,45,Software,,2
3,Nusantara Export,https://nusantaraexport.co.id,Budi Santoso,Owner,budi@nusantaraexport.co.id,Indonesia,Surabaya,30,Export,,3
4,Bright Path Interns,https://brightpath.example.com,Sam Lee,Marketing Intern,sam@brightpath.example.com,Malaysia,Penang,8,Agency,,4
5,Global Mega Corp,https://globalmega.example.com,Jane Doe,Director of Operations,jane.doe@globalmega.example.com,United States,Austin,12000,Manufacturing,,5
6,Hong Kong Trade Partners,https://hktradepartners.example.hk,Lam Ka Yiu,Managing Director,lam@hktradepartners.example.hk,Hong Kong,Kowloon,65,Distribution,,6
7,Broken Email Ltd,https://broken-email.example.com,No Name,COO,not-an-email,Vietnam,Hanoi,90,Software,,7
8,No Website Sdn Bhd,,Tan Mei Ling,General Manager,tan@nowebsite.example.my,Malaysia,Johor,55,Wholesale,,8
9,Dead Domain Industries,https://this-domain-does-not-resolve-xyz123.com,Ghost User,CTO,ghost@this-domain-does-not-resolve-xyz123.com,Thailand,Bangkok,200,Manufacturing,,9



Shape: (12, 11)


---
## Cell 5 — Stage 2: Clean

The important move here is **deduping on root domain, not company name**.
"Rui Feng Logistics" and "Rui Feng Logistics Co Ltd" are the same company; no
string-similarity trick catches every variant of that, but both resolve to
`ruifeng-logistics.com`. Domain is the reliable key.


In [8]:
# --- Cell 5: Stage 2 — Clean ---------------------------------------------
def norm_text(v):
    if pd.isna(v) or v is None:
        return None
    s = str(v).strip()
    s = re.sub(r"\s+", " ", s)
    return s or None

def norm_url(v):
    # Return a canonical https URL, or None if unusable.
    s = norm_text(v)
    if not s:
        return None
    if not s.startswith(("http://", "https://")):
        s = "https://" + s
    try:
        p = urlparse(s)
        if not p.netloc:
            return None
        netloc = p.netloc.lower().replace("www.", "")
        return urlunparse(("https", netloc, p.path.rstrip("/"), "", "", ""))
    except Exception:
        return None

def root_domain(v):
    # ruifeng-logistics.com from any URL or email
    s = norm_text(v)
    if not s:
        return None
    if "@" in s:
        s = s.split("@")[-1]
    ext = tldextract.extract(s)
    if not ext.domain or not ext.suffix:
        return None
    return f"{ext.domain}.{ext.suffix}".lower()

def norm_employees(v):
    # Handles 120, "120", "51-200", "1,200+", "" -> int or None
    if pd.isna(v) or v is None or str(v).strip() == "":
        return None
    s = str(v).replace(",", "")
    nums = re.findall(r"\d+", s)
    if not nums:
        return None
    nums = [int(n) for n in nums]
    return int(sum(nums) / len(nums)) if len(nums) > 1 else nums[0]


def clean(df):
    d = df.copy()

    for c in ["company_name", "contact_name", "job_title", "country", "city", "industry"]:
        d[c] = d[c].apply(norm_text)

    d["country"] = d["country"].apply(lambda x: x.lower() if x else None)
    d["email"] = d["email"].apply(lambda x: norm_text(x).lower() if norm_text(x) else None)
    d["website"] = d["website"].apply(norm_url)
    d["linkedin_url"] = d["linkedin_url"].apply(norm_url)
    d["employee_count"] = d["employee_count"].apply(norm_employees)

    # Domain key: prefer website, fall back to email domain.
    d["domain"] = d["website"].apply(root_domain)
    d.loc[d["domain"].isna(), "domain"] = d.loc[d["domain"].isna(), "email"].apply(root_domain)

    before = len(d)

    # Rows with neither a domain nor an email cannot be delivered. Park them.
    unusable = d[d["domain"].isna() & d["email"].isna()].copy()
    unusable["_drop_reason"] = "no domain and no email"
    d = d[~(d["domain"].isna() & d["email"].isna())].copy()

    # Dedupe: one row per (domain, email). Keep the row with the most filled fields.
    d["_completeness"] = d.notna().sum(axis=1)
    d = (d.sort_values("_completeness", ascending=False)
           .drop_duplicates(subset=["domain", "email"], keep="first")
           .drop(columns=["_completeness"])
           .sort_values("_source_row")
           .reset_index(drop=True))

    log.info(f"Clean: {before} -> {len(d)} rows "
             f"({before - len(d)} removed: {len(unusable)} unusable, "
             f"{before - len(d) - len(unusable)} duplicates)")
    return d, unusable


df_clean, df_dropped = clean(df_raw)
save_checkpoint(df_clean, "02_clean")

print("\nDropped rows:")
display(df_dropped[["company_name", "_drop_reason"]] if len(df_dropped) else "none")
display(df_clean[["company_name", "domain", "email", "job_title", "employee_count", "country"]])



Dropped rows:


'none'

,company_name,domain,email,job_title,employee_count,country
0,Rui Feng Logistics,ruifeng-logistics.com,chen.wei@ruifeng-logistics.com,Chief Executive Officer,120,taiwan
1,Meridian SaaS Pte Ltd,example.sg,aisyah@meridian-saas.example.sg,Head of Growth,45,singapore
2,Nusantara Export,nusantaraexport.co.id,budi@nusantaraexport.co.id,Owner,30,indonesia
3,Bright Path Interns,example.com,sam@brightpath.example.com,Marketing Intern,8,malaysia
4,Global Mega Corp,example.com,jane.doe@globalmega.example.com,Director of Operations,12000,united states
5,Hong Kong Trade Partners,example.hk,lam@hktradepartners.example.hk,Managing Director,65,hong kong
6,Broken Email Ltd,example.com,not-an-email,COO,90,vietnam
7,No Website Sdn Bhd,example.my,tan@nowebsite.example.my,General Manager,55,malaysia
8,Dead Domain Industries,this-domain-does-not-resolve-xyz123.com,ghost@this-domain-does-not-resolve-xyz123.com,CTO,200,thailand
9,上海精密制造有限公司,example.cn,zhang.wei@shanghai-precision.example.cn,总经理,180,china


---
## Cell 6 — Stage 3: Enrich

The slow stage. One HTTP request per company, throttled to one every
`REQUEST_DELAY_SEC` seconds. At the default 2s, 100 leads takes about 4 minutes.

Three things happen per company:
- **Language detection** by CJK character ratio. No extra dependency, and more
  reliable than statistical detectors on short marketing copy.
- **Industry keyword matching** against the actual site text, not the CRM's
  stale industry label.
- **Extra email discovery** from the page, for rows that arrived without one.

Results checkpoint every 25 rows. If Colab disconnects, re-run this cell — it
resumes from where it stopped rather than re-fetching everything.


In [9]:
# --- Cell 6: Stage 3 — Enrich --------------------------------------------
CJK_RE   = re.compile(r"[\u4e00-\u9fff\u3400-\u4dbf]")
EMAIL_RE = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}")

def detect_language(text):
    # Returns 'zh', 'en', or 'other' based on character composition.
    if not text or len(text) < 40:
        return None
    sample = text[:5000]
    cjk = len(CJK_RE.findall(sample))
    letters = sum(c.isalpha() for c in sample)
    if letters == 0:
        return None
    if cjk / max(letters, 1) > 0.15:
        return "zh"
    ascii_letters = sum(c.isascii() and c.isalpha() for c in sample)
    return "en" if ascii_letters / letters > 0.85 else "other"


def enrich_one(row):
    # Returns a dict of enrichment fields for a single lead.
    out = {
        "site_status": None, "site_title": None, "site_language": None,
        "industry_matches": None, "industry_match_count": 0,
        "discovered_emails": None, "site_text_chars": 0,
    }
    url = row.get("website")
    if not url:
        out["site_status"] = "no_website"
        return out

    html, status = fetch(url)
    out["site_status"] = status
    if html is None:
        return out

    try:
        soup = BeautifulSoup(html, "lxml")
        for tag in soup(["script", "style", "noscript"]):
            tag.decompose()
        text = soup.get_text(" ", strip=True)

        out["site_title"]      = (soup.title.string or "").strip()[:200] if soup.title else None
        out["site_text_chars"] = len(text)
        out["site_language"]   = detect_language(text)

        low = text.lower()
        hits = [k for k in CONFIG["ICP"]["industry_keywords"] if k in low]
        out["industry_matches"]     = ", ".join(hits) if hits else None
        out["industry_match_count"] = len(hits)

        found = set(EMAIL_RE.findall(html))
        found = {e.lower() for e in found
                 if not e.lower().endswith((".png", ".jpg", ".gif", ".svg", ".webp"))}
        out["discovered_emails"] = ", ".join(sorted(found)[:5]) if found else None
    except Exception as e:
        out["site_status"] = f"parse_error_{type(e).__name__}"
    return out


def enrich(df):
    d = df.copy()
    enrich_cols = ["site_status", "site_title", "site_language", "industry_matches",
                   "industry_match_count", "discovered_emails", "site_text_chars"]

    # Resume from an earlier partial run if one exists.
    partial = load_checkpoint("03_enriched_partial")
    if partial is not None and set(enrich_cols).issubset(partial.columns):
        d = d.merge(partial[["_source_row"] + enrich_cols], on="_source_row", how="left")
    else:
        for c in enrich_cols:
            d[c] = None

    todo = d[d["site_status"].isna()].index.tolist()
    if CONFIG["ENRICH_LIMIT"]:
        todo = todo[:CONFIG["ENRICH_LIMIT"]]

    log.info(f"Enriching {len(todo)} of {len(d)} rows "
             f"(~{len(todo) * CONFIG['REQUEST_DELAY_SEC'] / 60:.1f} min)")

    for n, idx in enumerate(tqdm(todo, desc="Enriching")):
        res = enrich_one(d.loc[idx].to_dict())
        for k, v in res.items():
            d.at[idx, k] = v
        time.sleep(CONFIG["REQUEST_DELAY_SEC"])
        if (n + 1) % 25 == 0:
            save_checkpoint(d, "03_enriched_partial")

    save_checkpoint(d, "03_enriched_partial")
    return d


df_enriched = enrich(df_clean)
save_checkpoint(df_enriched, "03_enriched")

print("\nFetch outcomes:")
print(df_enriched["site_status"].value_counts(dropna=False).to_string())
print("\nSite languages:")
print(df_enriched["site_language"].value_counts(dropna=False).to_string())
display(df_enriched[["company_name", "site_status", "site_language",
                     "industry_matches", "discovered_emails"]])


Enriching:   0%|          | 0/11 [00:00<?, ?it/s]


Fetch outcomes:
site_status
connection_error    7
ok                  1
no_website          1
timeout             1
ssl_error           1

Site languages:
site_language
None    10
en       1


,company_name,site_status,site_language,industry_matches,discovered_emails
0,Rui Feng Logistics,connection_error,None,None,None
1,Meridian SaaS Pte Ltd,connection_error,None,None,None
2,Nusantara Export,ok,en,export,"export@nusantaraexport.co.id, info@nusantaraex..."
3,Bright Path Interns,connection_error,None,None,None
4,Global Mega Corp,connection_error,None,None,None
5,Hong Kong Trade Partners,connection_error,None,None,None
6,Broken Email Ltd,connection_error,None,None,None
7,No Website Sdn Bhd,no_website,None,None,None
8,Dead Domain Industries,connection_error,None,None,None
9,上海精密制造有限公司,timeout,None,None,None


---
## Cell 7 — Stage 4: Verify

This is the stage that justifies charging for "verified" leads, so be precise
about what it does and does not prove.

**It checks:** syntax is valid, the domain is not a free/disposable provider,
and the domain publishes MX records — i.e. it can receive mail at all.

**It does not check:** whether that specific mailbox exists. Real per-mailbox
verification needs SMTP probing, which gets your IP blacklisted, or a paid API
(NeverBounce, ZeroBounce). Say `mx_valid` to clients, not "100% deliverable" —
overclaiming here is how lead gen freelancers lose accounts.


In [10]:
# --- Cell 7: Stage 4 — Verify --------------------------------------------
STRICT_EMAIL_RE = re.compile(r"^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$")

FREE_PROVIDERS = {
    "gmail.com","yahoo.com","hotmail.com","outlook.com","aol.com","icloud.com",
    "qq.com","163.com","126.com","sina.com","foxmail.com","yandex.com","gmx.com",
}
DISPOSABLE = {
    "mailinator.com","guerrillamail.com","10minutemail.com","tempmail.com",
    "throwawaymail.com","yopmail.com","trashmail.com",
}

_MX_CACHE = {}

def has_mx(domain):
    # True / False / None(lookup failed). Cached per domain.
    if not domain:
        return None
    if domain in _MX_CACHE:
        return _MX_CACHE[domain]
    try:
        answers = dns.resolver.resolve(domain, "MX", lifetime=5.0)
        result = len(answers) > 0
    except (dns.resolver.NXDOMAIN, dns.resolver.NoAnswer):
        result = False
    except Exception:
        result = None                      # timeout / server failure: unknown
    _MX_CACHE[domain] = result
    time.sleep(0.2)
    return result


def verify(df):
    d = df.copy()
    d["email_syntax_ok"] = d["email"].apply(
        lambda e: bool(STRICT_EMAIL_RE.match(e)) if e else False)

    d["email_domain"] = d["email"].apply(
        lambda e: e.split("@")[-1].lower() if e and "@" in e else None)

    d["is_free_provider"]  = d["email_domain"].apply(lambda x: x in FREE_PROVIDERS if x else False)
    d["is_disposable"]     = d["email_domain"].apply(lambda x: x in DISPOSABLE if x else False)

    uniq = sorted({x for x in d["email_domain"].dropna().unique()})
    log.info(f"MX lookup on {len(uniq)} unique domains")
    for dom in tqdm(uniq, desc="MX lookups"):
        has_mx(dom)
    d["mx_valid"] = d["email_domain"].apply(has_mx)

    def verdict(r):
        if not r["email"]:            return "no_email"
        if not r["email_syntax_ok"]:  return "invalid_syntax"
        if r["is_disposable"]:        return "disposable"
        if r["mx_valid"] is False:    return "no_mx"
        if r["mx_valid"] is None:     return "mx_unknown"
        if r["is_free_provider"]:     return "valid_free_provider"
        return "valid_business"

    d["email_verdict"] = d.apply(verdict, axis=1)
    # Only these two verdicts count as verified in the scorer.
    d["email_verified"] = d["email_verdict"].isin(["valid_business", "valid_free_provider"])

    log.info("Verification outcomes:\n" + d["email_verdict"].value_counts().to_string())
    return d


df_verified = verify(df_enriched)
save_checkpoint(df_verified, "04_verified")
display(df_verified[["company_name", "email", "email_verdict", "mx_valid", "email_verified"]])


MX lookups:   0%|          | 0/10 [00:00<?, ?it/s]

,company_name,email,email_verdict,mx_valid,email_verified
0,Rui Feng Logistics,chen.wei@ruifeng-logistics.com,no_mx,False,False
1,Meridian SaaS Pte Ltd,aisyah@meridian-saas.example.sg,no_mx,False,False
2,Nusantara Export,budi@nusantaraexport.co.id,no_mx,False,False
3,Bright Path Interns,sam@brightpath.example.com,no_mx,False,False
4,Global Mega Corp,jane.doe@globalmega.example.com,no_mx,False,False
5,Hong Kong Trade Partners,lam@hktradepartners.example.hk,no_mx,False,False
6,Broken Email Ltd,not-an-email,invalid_syntax,None,False
7,No Website Sdn Bhd,tan@nowebsite.example.my,no_mx,False,False
8,Dead Domain Industries,ghost@this-domain-does-not-resolve-xyz123.com,no_mx,False,False
9,上海精密制造有限公司,zhang.wei@shanghai-precision.example.cn,mx_unknown,None,False


---
## Cell 8 — Stage 5: Score

A 100-point rubric where **every point carries a written reason**. The
`score_reasons` column is what lets a client ask "why is this an 82?" and get a
straight answer instead of a shrug. That auditability is the differentiator —
most freelancers hand over an unexplained ranking.

Weights live in `CONFIG["WEIGHTS"]`. Change them there, never here.


In [11]:
# --- Cell 8: Stage 5 — Score ---------------------------------------------
W   = CONFIG["WEIGHTS"]
ICP = CONFIG["ICP"]

def title_norm(t):
    # "Chief Executive Officer (CEO)" -> "chief executive officer ceo"
    # "C.E.O."                        -> "ceo"      (dots dissolved, not split)
    # "总经理"                          -> "总经理"    (CJK preserved)
    if t is None or (isinstance(t, float) and pd.isna(t)):
        return ""
    s = str(t).lower()
    s = re.sub(r"[.\u2019']", "", s)                       # C.E.O. -> ceo
    s = re.sub(r"[^a-z0-9\s\u4e00-\u9fff\u3400-\u4dbf]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

_HAS_CJK = re.compile(r"[\u4e00-\u9fff\u3400-\u4dbf]")

def title_matches(title, keywords):
    # Latin keywords use word boundaries, so 'ceo' will not fire inside
    # 'ceonomics' and 'intern' will not fire inside 'internal audit'.
    # CJK has no word boundaries, so those keywords use substring matching.
    for kw in keywords:
        if _HAS_CJK.search(kw):
            if kw in title:
                return kw
        elif re.search(r"\b" + re.escape(kw) + r"\b", title):
            return kw
    return None


def score_one(r):
    # Reason-before-score: build the justification list, sum points from it.
    pts, reasons = 0, []

    # --- 1. Email verified ---
    if r["email_verdict"] == "valid_business":
        pts += W["email_verified"]
        reasons.append(f"+{W['email_verified']} business email with valid MX")
    elif r["email_verdict"] == "valid_free_provider":
        half = W["email_verified"] // 2
        pts += half
        reasons.append(f"+{half} valid email but free provider (weaker signal)")
    else:
        reasons.append(f"+0 email not verified ({r['email_verdict']})")

    # --- 2. Decision maker (exclusions checked first) ---
    title = title_norm(r.get("job_title"))
    if not title:
        reasons.append("+0 no title on record")
    elif (hit := title_matches(title, ICP["excluded_titles"])):
        reasons.append(f"+0 excluded title ('{hit}')")
    elif (hit := title_matches(title, ICP["decision_maker_titles"])):
        pts += W["decision_maker"]
        reasons.append(f"+{W['decision_maker']} decision-maker title ('{hit}')")
    elif (hit := title_matches(title, ICP["influencer_titles"])):
        half = W["decision_maker"] // 2
        pts += half
        reasons.append(f"+{half} influencer title ('{hit}'), not budget holder")
    else:
        reasons.append("+0 title present but unrecognised")

    # --- 3. Company size fit ---
    emp = r["employee_count"]
    if emp is None or pd.isna(emp):
        reasons.append("+0 employee count unknown")
    elif ICP["employee_min"] <= emp <= ICP["employee_max"]:
        pts += W["company_size_fit"]
        reasons.append(f"+{W['company_size_fit']} size {int(emp)} within ICP band")
    else:
        reasons.append(f"+0 size {int(emp)} outside ICP band")

    # --- 4. Industry match (site evidence preferred over CRM label) ---
    n = r.get("industry_match_count") or 0
    if n >= 2:
        pts += W["industry_match"]
        reasons.append(f"+{W['industry_match']} {n} industry keywords on site")
    elif n == 1:
        half = W["industry_match"] // 2
        pts += half
        reasons.append(f"+{half} 1 industry keyword on site")
    else:
        label = (r.get("industry") or "").lower()
        if label and any(k in label for k in ICP["industry_keywords"]):
            third = W["industry_match"] // 3
            pts += third
            reasons.append(f"+{third} industry from CRM label only (unverified)")
        else:
            reasons.append("+0 no industry match")

    # --- 5. APAC / Mandarin edge ---
    country = (r.get("country") or "").lower()
    if any(m in country for m in ICP["mandarin_markets"]) or r.get("site_language") == "zh":
        pts += W["apac_mandarin_edge"]
        reasons.append(f"+{W['apac_mandarin_edge']} Mandarin-market lead (outreach edge)")
    elif any(c in country for c in ICP["target_countries"]):
        half = W["apac_mandarin_edge"] // 2
        pts += half
        reasons.append(f"+{half} APAC target country")
    else:
        reasons.append("+0 outside target geography")

    # --- 6. Website live ---
    if r.get("site_status") == "ok":
        pts += W["website_live"]
        reasons.append(f"+{W['website_live']} website reachable")
    else:
        reasons.append(f"+0 website not reachable ({r.get('site_status')})")

    return pts, " | ".join(reasons)


def tier_of(score):
    t = CONFIG["TIERS"]
    if score >= t["A"]: return "A"
    if score >= t["B"]: return "B"
    if score >= t["C"]: return "C"
    return "D"


def score(df):
    d = df.copy()
    results = d.apply(score_one, axis=1)
    d["lead_score"]    = [x[0] for x in results]
    d["score_reasons"] = [x[1] for x in results]
    d["tier"]          = d["lead_score"].apply(tier_of)
    d = d.sort_values("lead_score", ascending=False).reset_index(drop=True)
    d["rank"] = range(1, len(d) + 1)

    log.info("Tier distribution:\n" + d["tier"].value_counts().sort_index().to_string())
    log.info(f"Mean score: {d['lead_score'].mean():.1f} | median: {d['lead_score'].median():.0f}")
    return d


df_scored = score(df_verified)
save_checkpoint(df_scored, "05_scored")
display(df_scored[["rank", "company_name", "lead_score", "tier", "email_verdict"]].head(20))

print("\nWorked example — why the top lead scored what it did:")
top = df_scored.iloc[0]
print(f"{top['company_name']}: {top['lead_score']}/100 (tier {top['tier']})")
for reason in top["score_reasons"].split(" | "):
    print("   " + reason)


,rank,company_name,lead_score,tier,email_verdict
0,1,Taipei Software Group,80,A,valid_business
1,2,Nusantara Export,57,B,no_mx
2,3,Rui Feng Logistics,50,C,no_mx
3,4,Meridian SaaS Pte Ltd,50,C,no_mx
4,5,Hong Kong Trade Partners,50,C,no_mx
5,6,上海精密制造有限公司,50,C,mx_unknown
6,7,Broken Email Ltd,45,C,invalid_syntax
7,8,Dead Domain Industries,45,C,no_mx
8,9,No Website Sdn Bhd,45,C,no_mx
9,10,Global Mega Corp,25,D,no_mx



Worked example — why the top lead scored what it did:
Taipei Software Group: 80/100 (tier A)
   +30 business email with valid MX
   +20 decision-maker title ('ceo')
   +15 size 75 within ICP band
   +5 industry from CRM label only (unverified)
   +10 Mandarin-market lead (outreach edge)
   +0 website not reachable (ssl_error)


---
## Cell 9 — Measuring whether the score actually works

Right now the rubric is a **hypothesis**, not a validated model. It is built on
plausible reasoning, and plausible reasoning is often wrong.

This cell does nothing useful until you have outcome data — a `replied` or
`booked` column from a campaign that has actually run. Once you do, it compares
your score against a **named baseline: random ordering**. If your top-decile
reply rate is not clearly above the random rate, the rubric is decoration.

Do not put lift numbers in a proposal before running this. Run 200 leads,
measure, then quote the real figure.


In [12]:
# --- Cell 9: validation (needs outcome data) ------------------------------
OUTCOME_COL = "replied"     # set to your outcome column once you have one

def validate(df, outcome_col=OUTCOME_COL, k_pct=0.20, n_bootstrap=1000):
    if outcome_col not in df.columns:
        print(f"No '{outcome_col}' column yet — nothing to validate.")
        print("Run a campaign, join the replies back on 'email', then re-run this cell.")
        return None

    d = df.dropna(subset=[outcome_col]).copy()
    d[outcome_col] = d[outcome_col].astype(int)
    if len(d) < 30:
        print(f"Only {len(d)} labelled rows. Too few to conclude anything. Need 100+.")
        return None

    import numpy as np
    k = max(1, int(len(d) * k_pct))
    top_k_rate = d.nlargest(k, "lead_score")[outcome_col].mean()
    overall    = d[outcome_col].mean()

    # Baseline: shuffle the ranking, take the same top-k, repeat.
    rng = np.random.default_rng(42)
    vals = d[outcome_col].values
    random_rates = [rng.permutation(vals)[:k].mean() for _ in range(n_bootstrap)]
    baseline = float(np.mean(random_rates))
    p95      = float(np.percentile(random_rates, 95))

    print(f"Labelled leads:            {len(d)}")
    print(f"Top {int(k_pct*100)}% by score (n={k}):  {top_k_rate:.1%} reply rate")
    print(f"Random baseline:           {baseline:.1%} (95th pct: {p95:.1%})")
    print(f"Overall rate:              {overall:.1%}")
    print(f"Lift vs baseline:          {top_k_rate / baseline:.2f}x" if baseline > 0 else "")
    print()
    if top_k_rate > p95:
        print("Score beats random beyond the 95th percentile. The rubric is doing work.")
    else:
        print("Score is inside random noise. Reweight before quoting any lift figure.")
    return {"top_k_rate": top_k_rate, "baseline": baseline, "n": len(d)}


validate(df_scored)


No 'replied' column yet — nothing to validate.
Run a campaign, join the replies back on 'email', then re-run this cell.


---
## Cell 10 — Stage 6: Export

Builds the client deliverable: a five-tab XLSX matching the structure you used
on the Allianz file, with a formatted summary tab and frozen headers.

Note the **Scored** tab excludes internal columns. Clients get the ranking and
the reasons, not your intermediate scraping fields.


In [13]:
# --- Cell 10: Stage 6 — Export -------------------------------------------
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)

CLIENT_COLS = ["rank", "tier", "lead_score", "company_name", "website", "domain",
               "contact_name", "job_title", "email", "email_verdict",
               "country", "city", "employee_count", "industry",
               "site_language", "linkedin_url", "score_reasons"]

def build_summary(df):
    rows = [
        ("Client",              CONFIG["client_name"]),
        ("Campaign",            CONFIG["campaign_name"]),
        ("Generated (UTC)",     datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M")),
        ("Run ID",              RUN_ID),
        ("", ""),
        ("Raw rows ingested",   len(df_raw)),
        ("After cleaning",      len(df_clean)),
        ("Rows dropped",        len(df_raw) - len(df_clean)),
        ("Delivered leads",     len(df)),
        ("", ""),
        ("Tier A (>=%d)" % CONFIG["TIERS"]["A"], int((df["tier"] == "A").sum())),
        ("Tier B (>=%d)" % CONFIG["TIERS"]["B"], int((df["tier"] == "B").sum())),
        ("Tier C (>=%d)" % CONFIG["TIERS"]["C"], int((df["tier"] == "C").sum())),
        ("Tier D (below)",                        int((df["tier"] == "D").sum())),
        ("", ""),
        ("Business emails, MX valid", int((df["email_verdict"] == "valid_business").sum())),
        ("Free-provider emails",      int((df["email_verdict"] == "valid_free_provider").sum())),
        ("Unverified / no email",     int((~df["email_verified"]).sum())),
        ("", ""),
        ("Websites reachable",        int((df["site_status"] == "ok").sum())),
        ("Chinese-language sites",    int((df["site_language"] == "zh").sum())),
        ("Mean score",                round(float(df["lead_score"].mean()), 1)),
        ("", ""),
        ("Verification method", "Syntax + domain MX record. Not per-mailbox SMTP."),
        ("Notes",               CONFIG["run_notes"]),
    ]
    return pd.DataFrame(rows, columns=["Metric", "Value"])


def export(df):
    fname = f"{CONFIG['campaign_name']}_leads_{RUN_ID}.xlsx"
    path  = os.path.join(CONFIG["OUTPUT_DIR"], fname)

    client_view = df[[c for c in CLIENT_COLS if c in df.columns]].copy()

    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        build_summary(df).to_excel(xw, sheet_name="Summary", index=False)
        client_view.to_excel(xw, sheet_name="Scored Leads", index=False)
        client_view[client_view["tier"].isin(["A", "B"])].to_excel(
            xw, sheet_name="Priority (A-B)", index=False)
        df_dropped.to_excel(xw, sheet_name="Rejected", index=False)
        df_raw.to_excel(xw, sheet_name="Raw Source", index=False)

        head_font = Font(bold=True, color="FFFFFF", size=11)
        head_fill = PatternFill("solid", start_color="2F5597")

        for sheet in xw.book.worksheets:
            for cell in sheet[1]:
                cell.font, cell.fill = head_font, head_fill
                cell.alignment = Alignment(horizontal="left", vertical="center")
            sheet.freeze_panes = "A2"
            for col in sheet.columns:
                letter = get_column_letter(col[0].column)
                width = max((len(str(c.value)) for c in col if c.value), default=10)
                sheet.column_dimensions[letter].width = min(max(width + 2, 12), 55)

    log.info(f"Exported: {path}")
    return path, client_view


output_path, client_view = export(df_scored)

# --- optional: push to Google Sheets ---
if CONFIG["WRITE_TO_SHEET"] and CONFIG["OUTPUT_SHEET_ID"]:
    from google.colab import auth
    import gspread
    from gspread_dataframe import set_with_dataframe
    from google.auth import default
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)
    sh = gc.open_by_key(CONFIG["OUTPUT_SHEET_ID"])
    for tab, frame in [("Scored Leads", client_view), ("Summary", build_summary(df_scored))]:
        try:
            ws = sh.worksheet(tab)
            ws.clear()
        except Exception:
            ws = sh.add_worksheet(title=tab, rows=len(frame) + 10, cols=len(frame.columns) + 5)
        set_with_dataframe(ws, frame)
    log.info("Pushed to Google Sheet.")

# --- download ---
try:
    from google.colab import files
    files.download(output_path)
except Exception:
    print(f"Not in Colab. File is at: {output_path}")

display(build_summary(df_scored))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,Metric,Value
0,Client,Demo Client
1,Campaign,apac_saas_q3
2,Generated (UTC),2026-08-03 14:33
3,Run ID,20260803_142908
4,,
5,Raw rows ingested,12
6,After cleaning,11
7,Rows dropped,1
8,Delivered leads,11
9,,


---
## Cell 11 — Known failure modes

Written down deliberately. When a client asks what the pipeline misses, having
this list ready is worth more than pretending it misses nothing.

| Failure mode | Effect | Mitigation |
|---|---|---|
| JS-rendered sites | `site_text_chars` near zero, industry score wrongly 0 | Check the low-text rows by hand; Playwright if it becomes common |
| Cloudflare / bot walls | `http_403`, website points lost | Treat 403 as unknown, not absent; verify by hand |
| Catch-all mail domains | `mx_valid` True but mailbox may not exist | Say "MX valid", never "deliverable" |
| MX timeout | `mx_unknown` scores 0, penalising a possibly good lead | Re-run Cell 7; the cache keeps it cheap |
| Parked domains | Site resolves, content is a placeholder | Low `site_text_chars` plus no keyword hits flags these |
| Company rebrand | Old domain 404s, lead looks dead | Search the company name before discarding |
| Multiple contacts, one company | Domain dedupe may drop a better contact | Dedupe key is (domain, email), so distinct people survive |
| Rubric never validated | Confident ranking, no evidence | Cell 9 against a random baseline before quoting lift |

**Two things to keep straight for client work:**

1. `RESPECT_ROBOTS` is `True` by default. Leave it. A blacklisted client domain
   costs far more than the leads you would gain.
2. For EU or UK contacts, GDPR applies to B2B personal data. Legitimate interest
   can cover B2B outreach, but you need a lawful basis and a working opt-out.
   The `target_countries` default is APAC-only, which sidesteps this — widen it
   deliberately, not accidentally.


---
## Next steps

1. Run top to bottom on `INPUT_MODE = "sample"` and confirm the export downloads.
2. Switch to `csv_upload` with 25 real rows, `ENRICH_LIMIT = 25`. Read every row
   of the output by hand and check the score reasons match your judgment.
3. Adjust `WEIGHTS` where they disagree with you. This is where the rubric gets good.
4. Remove `ENRICH_LIMIT`, run the full list.
5. After the first campaign, join reply data back and run Cell 9.

**Save your work:** File → Save a copy in GitHub, into your
`claude-agent-suite` repo. Colab wipes `/content` on disconnect; the notebook
itself is only safe once it is in Drive or GitHub.
